In [4]:
import pandas as pd
import numpy as np
from anjana.anonymity import (
    k_anonymity_inner,
    k_anonymity,
    l_diversity,
    t_closeness,
    alpha_k_anonymity,
)
import masking_effects_on_xai_techniques.hierarchies as hi

In [5]:
path = "../data/adult/"

In [6]:
data = pd.read_csv(path + "data.csv")
len_data = len(data)

# Clean the data
data["income"] = data["income"].str.rstrip(".")
data.dropna(inplace=True)
data.drop(columns=["education-num", "fnlwgt"], inplace=True)

len_clean_data = len(data)
print(f"Dropped {len_data - len_clean_data} rows")
data.to_csv(path + "clean.csv", index=False)

Dropped 1221 rows


In [7]:
data.shape

(47621, 13)

In [8]:
hierarchies_path = "../hierarchies/adult"

for feat in ["age", "capital-gain", "capital-loss", "hours-per-week"]:
    hiear = hi.generate_qcut_hierarchy(data[feat], 7, search_for_n=True)
    hi.save_hierarchy(hiear, f"{hierarchies_path}/{feat}.csv")

hierarchies = {
    # "age": dict(pd.read_csv(f"{hierarchies_path}/age.csv", header=None)),
    "education": dict(pd.read_csv(f"{hierarchies_path}/education.csv", header=None)),
    "marital-status": dict(
        pd.read_csv(f"{hierarchies_path}/marital-status.csv", header=None)
    ),
    "occupation": dict(pd.read_csv(f"{hierarchies_path}/occupation.csv", header=None)),
    "sex": dict(pd.read_csv(f"{hierarchies_path}/sex.csv", header=None)),
    "native-country": dict(
        pd.read_csv(f"{hierarchies_path}/native-country.csv", header=None)
    ),
    "workclass": dict(pd.read_csv(f"{hierarchies_path}/workclass.csv", header=None)),
    "relationship": dict(
        pd.read_csv(f"{hierarchies_path}/relationship.csv", header=None)
    ),
    "race": dict(pd.read_csv(f"{hierarchies_path}/race.csv", header=None)),
    "capital-gain": dict(
        pd.read_csv(f"{hierarchies_path}/capital-gain.csv", header=None)
    ),
    "capital-loss": dict(
        pd.read_csv(f"{hierarchies_path}/capital-loss.csv", header=None)
    ),
    "hours-per-week": dict(
        pd.read_csv(f"{hierarchies_path}/hours-per-week.csv", header=None)
    ),
}

/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)


In [29]:
quasi_ident = list(data.columns)
quasi_ident.remove("race")
quasi_ident.remove("income")

ident = [
    "race"
]  # Race is marked as an indentifier, because it is identical across so many instances
sens_att = "income"

In [33]:
all_counts_and_features = []
for feat, items in hierarchies.items():
    for item in items.values():
        length = len(item.unique())
        all_counts_and_features.append((length, feat))
sorted_data = sorted(all_counts_and_features, key=lambda x: x[0], reverse=True)
ordered_features = [feat for count, feat in sorted_data]

for feat, items in hierarchies.items():
    lengths = [f"{len(item.unique()):>4}" for item in items.values()]
    out = ", ".join(lengths)
    print(f"{feat:<15}: {out}")

print("Order in which features will be generalized:")
print(ordered_features)

age            :   74,   14,    8,    5,    3,    2,    1
education      :   16,    5,    3,    1
marital-status :    7,    2,    1
occupation     :   15,    3,    1
sex            :    2,    1
native-country :   42,    7,    1
workclass      :    9,    4,    1
relationship   :    6,    1
race           :    5,    1
capital-gain   :  122,    7,    6,    5,    4,    3,    2,    1
capital-loss   :   98,    7,    6,    5,    4,    3,    2,    1
hours-per-week :   96,    7,    6,    5,    4,    3,    2,    1
Order in which features will be generalized:
['capital-gain', 'capital-loss', 'hours-per-week', 'age', 'native-country', 'education', 'occupation', 'age', 'workclass', 'age', 'marital-status', 'native-country', 'capital-gain', 'capital-loss', 'hours-per-week', 'relationship', 'capital-gain', 'capital-loss', 'hours-per-week', 'age', 'education', 'race', 'capital-gain', 'capital-loss', 'hours-per-week', 'workclass', 'capital-gain', 'capital-loss', 'hours-per-week', 'age', 'education', 'o

In [38]:
%%time
# Check with mina if this k is representative of used k in litterature
k = 10
supp_level = 20  # Select the suppression limit allowed
anon_df, supp_n, hiear = k_anonymity_inner(
    data, ident, quasi_ident, k, supp_level, hierarchies
)
max_supp_n = int(round(len(data) * supp_level / 100, 0))
print(f"Max rows that can be suppressed: {max_supp_n}")
print(f"Rows suppressed                : {supp_n}")
print(f"% of allowed rows suppressed   : {round(supp_n / max_supp_n * 100, 1)}%")
print(f"Generalization level           : {sum(hiear.values())}")
hiear

Max rows that can be suppressed: 9524
Rows suppressed                : 7376
% of allowed rows suppressed   : 77.4%
Generalization level           : 13
CPU times: user 10.1 s, sys: 91.1 ms, total: 10.2 s
Wall time: 10.2 s


{'age': 3,
 'workclass': 1,
 'education': 1,
 'marital-status': 1,
 'occupation': 1,
 'relationship': 1,
 'sex': 0,
 'capital-gain': 1,
 'capital-loss': 1,
 'hours-per-week': 1,
 'native-country': 2}